# Cognitive Digital Twin-Driven Multi-Agent Federated Reinforcement Learning (CDT-FedMARL)
## Autonomous Resource Orchestration in the Cloud–Edge Continuum — Google Colab implementation

**Pipeline**
1. Load a real cluster workload trace (Alibaba Cluster Trace v2018 by default; Google / Azure / Bitbrains / own CSV supported) or a synthetic diurnal trace as fallback.
2. **Cognitive Digital Twin (CDT)** per edge node: a GRU forecaster trained *federatedly* (FedAvg) that predicts next-slot background load, plus a *what-if* simulator that estimates latency of every offloading option before the agent acts (feature augmentation + risk-aware action masking).
3. **Cloud–edge continuum environment**: K edge nodes + 1 cloud; each edge agent decides *where* to run its task batch (local / neighbour edge / cloud) and *at what CPU frequency* (DVFS) → 9 discrete actions.
4. **Multi-agent Double-DQN** agents, one per edge node, whose Q-networks are aggregated with **FedAvg** (raw data never leaves the node).
5. Baselines + ablations: Random, Local-only, Cloud-only, Greedy heuristic, Independent MADRL, Fed-MADRL (no twin), DT-MADRL (no federation), **CDT-FedMARL (proposed)**.
6. Metrics: average latency, energy, SLA-violation rate, reward, twin forecast MAE/RMSE; figures + CSV tables saved to `results/`.

> Runtime → *Change runtime type* → CPU is enough (GPU optional). Default config runs in roughly 15–25 min.

In [ ]:
# ==== 1. Setup ====
import os, math, random, time, copy, json, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("results", exist_ok=True)
os.makedirs("data", exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

CFG = dict(
    seed=42,
    # ---- data ----
    data_source="alibaba",      # "alibaba" (falls back to "zenodo") | "zenodo" | "custom" | "synthetic"
    custom_csv="data/my_trace.csv",   # for "custom": columns node,timestamp,cpu (cpu in %)
    alibaba_max_lines=3_000_000,       # lines streamed from the Alibaba tarball (keeps Colab RAM low)
    resample="1min",
    # ---- system ----
    n_edge=6,                    # number of edge nodes = number of agents
    steps_per_ep=200,            # decision slots per episode
    edge_cap=20.0,               # Gcycles/s at max frequency
    cloud_cap=60.0,              # Gcycles/s
    bw_edge=200.0,               # Mb/s edge-edge
    bw_wan=100.0,                # Mb/s edge-cloud
    wan_rtt=40.0,                # ms base RTT to cloud
    hop_ms=4.0,                  # ms edge-edge hop
    lam_max=8.0,                 # max tasks per slot at 100 % load
    deadline_ms=300.0,           # SLA deadline per batch
    f_levels=[0.6, 0.8, 1.0],    # DVFS levels (fraction of max freq)
    w_lat=1.0, w_energy=0.5, w_sla=2.0,
    # ---- twin ----
    hist_len=12, twin_hidden=32, twin_rounds=20, twin_local_epochs=2, twin_lr=3e-3,
    twin_mask=True, mask_factor=1.5,
    # ---- RL ----
    episodes=200, gamma=0.95, lr=1e-3, batch=64, buffer=30000,
    eps_start=1.0, eps_end=0.05, eps_decay_frac=0.6,
    learn_every=2, target_every=300, hidden=128,
    fed_every=5,                 # FedAvg round every N episodes
    eval_episodes=10,
)
set_seed(CFG["seed"])
print("Device:", DEVICE)

## 2. Workload traces (data availability)
| Dataset | Link | Used for |
|---|---|---|
| Alibaba Cluster Trace v2018 | https://github.com/alibaba/clusterdata | per-machine CPU utilisation (default) |
| Google Cluster Workload Traces 2019 / 2011 | https://github.com/google/cluster-data | alternative (BigQuery / GCS) |
| Azure Public Dataset (VM traces) | https://github.com/Azure/AzurePublicDataset | alternative VM CPU readings |
| Bitbrains GWA-T-12 | http://gwa.ewi.tudelft.nl/datasets/gwa-t-12-bitbrains | alternative, small & easy |
| Zenodo mirror (Alibaba 2018 + Google 2019 + Azure v2, CC-BY 4.0) | https://doi.org/10.5281/zenodo.14564935 | automatic fallback, no survey needed |
| EUA edge server locations | https://github.com/swinedge/eua-dataset | optional edge topology |

The Alibaba loader streams only the first `alibaba_max_lines` rows of `machine_usage.tar.gz`, so Colab does not need to store the full archive. For Google/Azure/Bitbrains, export a CSV with columns `node,timestamp,cpu` and use `data_source="custom"`. If the Alibaba server is unreachable (it now requires a survey), the notebook automatically uses the open Zenodo mirror, building each edge site from a real data-centre CPU trace; only if that also fails does it use a synthetic trace.

In [ ]:
# ==== 2. Data loading ====
ALIBABA_URL = "http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2018Traces/machine_usage.tar.gz"
ALIBABA_COLS = ["machine_id", "time_stamp", "cpu_util_percent", "mem_util_percent",
                "mem_gps", "mkpi", "net_in", "net_out", "disk_io_percent"]

def to_matrix(df, n_nodes, resample):
    """df: columns node,timestamp(seconds),cpu(%) -> T x K matrix in [0,1]."""
    df = df.dropna(subset=["cpu"])
    top = df["node"].value_counts().index[:n_nodes]
    series = []
    for m in top:
        s = df[df["node"] == m].sort_values("timestamp")
        s = s.set_index(pd.to_timedelta(s["timestamp"] - s["timestamp"].min(), unit="s"))["cpu"]
        s = s.resample(resample).mean().interpolate().bfill().ffill()
        series.append(s.values)
    L = min(len(x) for x in series)
    M = np.stack([x[:L] for x in series], 1) / 100.0
    return np.clip(M, 0, 1)

def load_alibaba(n_nodes, max_lines, resample):
    out = "data/machine_usage_sample.csv"
    if not os.path.exists(out) or os.path.getsize(out) < 1000:
        print("Streaming Alibaba machine_usage (partial)...")
        cmd = f"curl -sL {ALIBABA_URL} | tar -xzO 2>/dev/null | head -n {max_lines} > {out}"
        subprocess.run(cmd, shell=True, timeout=1800)
    df = pd.read_csv(out, header=None, names=ALIBABA_COLS)
    df = df.rename(columns={"machine_id": "node", "time_stamp": "timestamp", "cpu_util_percent": "cpu"})
    return to_matrix(df[["node", "timestamp", "cpu"]], n_nodes, resample)

def load_custom(path, n_nodes, resample):
    df = pd.read_csv(path)
    return to_matrix(df[["node", "timestamp", "cpu"]], n_nodes, resample)

ZENODO = {  # DOI 10.5281/zenodo.14564935 (CC-BY 4.0), 300-second aggregates
    "alibaba2018": "https://zenodo.org/records/14564935/files/machine_usage_days_1_to_8_grouped_300_seconds.csv?download=1",
    "google2019":  "https://zenodo.org/records/14564935/files/instance_usage_grouped_300_seconds_month.csv?download=1",
    "azure_v2":    "https://zenodo.org/records/14564935/files/vm_cpu_readings_month_aggregated_cpu_mem.csv?download=1",
}

def _cpu_series(path):
    df = pd.read_csv(path)
    cols = [c for c in df.columns if "cpu" in c.lower()]
    if not cols:                                   # header-less file: take first numeric column
        df = pd.read_csv(path, header=None); cols = [df.select_dtypes("number").columns[-1]]
    x = pd.to_numeric(df[cols[0]], errors="coerce").interpolate().bfill().ffill().values.astype(float)
    x = x[(x >= 0)] if (x >= 0).any() else x
    return 100 * (x - x.min()) / (x.max() - x.min() + 1e-9)     # min-max to 0-100 %

def load_zenodo(n_nodes):
    """Builds n_nodes edge sites from three real data-centre traces (Alibaba, Google, Azure).
    Sites beyond 3 reuse a trace with an 8-hour circular time shift (= a site in another time zone)."""
    base = []
    for name, url in ZENODO.items():
        p = f"data/{name}.csv"
        if not os.path.exists(p) or os.path.getsize(p) < 1000:
            subprocess.run(f"curl -sL --max-time 300 -o {p} '{url}'", shell=True)
        base.append((name, _cpu_series(p)))
    L = min(len(x) for _, x in base)
    rows = []
    for k in range(n_nodes):
        name, x = base[k % len(base)]
        x = np.roll(x[:L], 96 * (k // len(base)))       # 96 x 5 min = 8 h shift
        rows.append(pd.DataFrame({"node": f"{name}_site{k}", "timestamp": np.arange(L) * 300, "cpu": x}))
    df = pd.concat(rows); df.to_csv("data/zenodo_edge_trace.csv", index=False)
    return to_matrix(df, n_nodes, "5min")

def synthetic_trace(n_nodes, T=2304, seed=0):
    """8 days of 5-min slots: diurnal + weekly + AR noise + bursts, node-specific phase."""
    rng = np.random.default_rng(seed)
    t = np.arange(T)
    M = np.zeros((T, n_nodes))
    for k in range(n_nodes):
        ph = rng.uniform(0, 2 * np.pi)
        base = 0.45 + 0.25 * np.sin(2 * np.pi * t / 288 + ph) + 0.05 * np.sin(2 * np.pi * t / 2016)
        ar = np.zeros(T)
        for i in range(1, T):
            ar[i] = 0.9 * ar[i - 1] + rng.normal(0, 0.03)
        bursts = (rng.random(T) < 0.01).astype(float)
        bursts = np.convolve(bursts, np.ones(6) * 0.25, mode="same")
        M[:, k] = base + ar + bursts
    return np.clip(M, 0.02, 1.0)

def load_trace(cfg):
    """Tries the requested source; 'alibaba' falls back to the Zenodo mirror, then synthetic."""
    src = cfg["data_source"]
    order = {"alibaba": ["alibaba", "zenodo", "synthetic"], "zenodo": ["zenodo", "synthetic"]}.get(
        src, [src, "synthetic"])
    for s_ in order:
        try:
            if s_ == "alibaba":
                M = load_alibaba(cfg["n_edge"], cfg["alibaba_max_lines"], cfg["resample"])
            elif s_ == "zenodo":
                M = load_zenodo(cfg["n_edge"])
            elif s_ in ("custom", "google", "azure", "bitbrains"):
                M = load_custom(cfg["custom_csv"], cfg["n_edge"], cfg["resample"])
            else:
                M = synthetic_trace(cfg["n_edge"], seed=cfg["seed"])
            if M.shape[0] < 4 * cfg["steps_per_ep"] or M.shape[1] < cfg["n_edge"]:
                raise ValueError(f"trace too short/narrow {M.shape}")
            if s_ != src: print(f"[fallback] '{src}' unavailable -> using '{s_}'")
            cfg["data_source"] = s_
            print(f"Loaded '{s_}' trace: {M.shape[0]} slots x {M.shape[1]} nodes")
            return M.astype(np.float32)
        except Exception as e:
            print(f"  could not load '{s_}': {e}")

TRACE = load_trace(CFG)
split = int(0.7 * len(TRACE))
TRAIN_TRACE, TEST_TRACE = TRACE[:split], TRACE[split:]

plt.figure(figsize=(10, 3))
for k in range(min(4, TRACE.shape[1])):
    plt.plot(TRACE[:600, k], lw=0.8, label=f"node {k}")
plt.axvline(min(split, 600), c="k", ls="--", lw=0.8)
plt.title(f"Background CPU load ({CFG['data_source']})"); plt.xlabel("slot"); plt.ylabel("utilisation")
plt.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.savefig("results/fig_trace.png", dpi=200); plt.show()

## 3. Cognitive Digital Twin — federated GRU load forecaster
Each edge node owns a twin forecaster trained on its own history only; FedAvg merges the twins every round.

In [ ]:
# ==== 3. Cognitive Digital Twin forecaster ====
class TwinGRU(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):                    # x: B x H  -> residual forecast on top of last value
        h, _ = self.gru(x.unsqueeze(-1))
        delta = 0.3 * torch.tanh(self.head(h[:, -1])).squeeze(-1)
        return torch.clamp(x[:, -1] + delta, 0.0, 1.0)

def windows(series, H):
    X = np.lib.stride_tricks.sliding_window_view(series[:-1], H)
    y = series[H:]
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def fedavg_state(states, weights=None):
    weights = np.ones(len(states)) if weights is None else np.asarray(weights, float)
    weights = weights / weights.sum()
    avg = copy.deepcopy(states[0])
    for k in avg:
        avg[k] = sum(w * s[k].float() for w, s in zip(weights, states))
    return avg

def train_federated_twin(train, cfg):
    H, K = cfg["hist_len"], train.shape[1]
    glob = TwinGRU(cfg["twin_hidden"]).to(DEVICE)
    data = [windows(train[:, k], H) for k in range(K)]
    hist = []
    for r in range(cfg["twin_rounds"]):
        states, sizes, losses = [], [], []
        for k in range(K):
            local = copy.deepcopy(glob); opt = torch.optim.Adam(local.parameters(), lr=cfg["twin_lr"])
            X, y = data[k]
            for _ in range(cfg["twin_local_epochs"]):
                perm = torch.randperm(len(X))
                for i in range(0, len(X), 256):
                    idx = perm[i:i + 256]
                    loss = F.mse_loss(local(X[idx].to(DEVICE)), y[idx].to(DEVICE))
                    opt.zero_grad(); loss.backward(); opt.step()
            states.append(local.state_dict()); sizes.append(len(X)); losses.append(loss.item())
        glob.load_state_dict(fedavg_state(states, sizes))
        hist.append(np.mean(losses))
        if (r + 1) % 5 == 0: print(f"twin round {r+1}: local MSE {hist[-1]:.5f}")
    return glob.eval(), hist

TWIN, twin_hist = train_federated_twin(TRAIN_TRACE, CFG)

# forecast accuracy on held-out slots vs. persistence baseline
with torch.no_grad():
    rows = []
    for k in range(TEST_TRACE.shape[1]):
        X, y = windows(TEST_TRACE[:, k], CFG["hist_len"])
        p = TWIN(X.to(DEVICE)).cpu().numpy(); y = y.numpy(); pers = X[:, -1].numpy()
        rows.append([k, np.abs(p - y).mean(), np.sqrt(((p - y) ** 2).mean()),
                     np.abs(pers - y).mean(), np.sqrt(((pers - y) ** 2).mean())])
twin_df = pd.DataFrame(rows, columns=["node", "Twin_MAE", "Twin_RMSE", "Persist_MAE", "Persist_RMSE"])
twin_df = pd.concat([twin_df, pd.DataFrame([["mean"] + list(twin_df.iloc[:, 1:].mean())], columns=twin_df.columns)], ignore_index=True)
twin_df.to_csv("results/table_twin_forecast.csv", index=False)
print(twin_df.round(4))

plt.figure(figsize=(10, 3))
plt.plot(y[:300], label="actual", lw=1); plt.plot(p[:300], label="CDT forecast", lw=1)
plt.title("Digital-twin one-step load forecast (test, last node)"); plt.legend(); plt.tight_layout()
plt.savefig("results/fig_twin_forecast.png", dpi=200); plt.show()

## 4. Cloud–edge continuum environment
Action `a ∈ {0..8}`: target = `a // 3` (0 local, 1 neighbour edge, 2 cloud), DVFS level = `f_levels[a % 3]`.
Agents see the **last measured** load; execution happens under the **next** load, so a good predictor (the twin) matters.

In [ ]:
# ==== 4. Environment ====
N_ACT = 9

class CloudEdgeEnv:
    def __init__(self, trace, cfg, twin=None, use_twin=False, seed=0):
        self.tr, self.c, self.twin, self.use_twin = trace, cfg, twin, use_twin
        self.K, self.H = trace.shape[1], cfg["hist_len"]
        self.f = np.array(cfg["f_levels"])
        self.rng = np.random.default_rng(seed)
        self.obs_dim = 10 + (6 if use_twin else 0)

    # ---------- dynamics helpers ----------
    def avail(self, j, load, f):          # usable Gcycles/s at node j
        return self.c["edge_cap"] * f * (1 - 0.8 * load[j]) + 1e-3

    def est_latency(self, i, target, f, load, rtt, Q=None):
        """Latency (ms) model; also used by the twin's what-if simulator."""
        Q = self.Q if Q is None else Q
        W, D = self.W[i], self.D[i]
        if target == 0:
            return 1000 * (Q[i] + W) / self.avail(i, load, f)
        if target == 1:
            j = (i + 1) % self.K
            return self.c["hop_ms"] + 1000 * D / self.c["bw_edge"] + 1000 * (Q[j] + W) / self.avail(j, load, f)
        return rtt + 1000 * D / self.c["bw_wan"] + 1000 * W / self.c["cloud_cap"]

    def energy(self, target, f, W, D):
        if target == 0: return 0.9 * f ** 2 * W                     # DVFS: E ∝ f² per cycle
        if target == 1: return 0.9 * f ** 2 * W + 0.05 * D
        return 0.15 * W + 0.5 * D                                     # radio tx energy + cloud share

    # ---------- episode ----------
    def reset(self, start=None):
        T = len(self.tr)
        lo, hi = self.H + 1, T - self.c["steps_per_ep"] - 2
        self.t = self.rng.integers(lo, hi) if start is None else start
        self.t0 = self.t
        self.Q = np.zeros(self.K)
        self.rtt = self.c["wan_rtt"]
        self._arrivals()
        return self._obs()

    def _arrivals(self):
        load = self.tr[self.t]                        # arrivals follow the current workload intensity
        n = self.rng.poisson(self.c["lam_max"] * load + 1)
        self.D = n * self.rng.uniform(0.5, 2.0, self.K)       # Mb
        self.W = self.D * self.rng.uniform(0.3, 0.6, self.K)  # Gcycles

    def _twin_features(self):
        H = self.H
        hist = torch.tensor(self.tr[self.t - H:self.t].T, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            pred = self.twin(hist).cpu().numpy()     # predicted load for slot t (unknown to agents)
        self.pred_load = np.clip(pred, 0, 1)
        feats = []
        for i in range(self.K):
            j = (i + 1) % self.K
            lat = [self.est_latency(i, a, 1.0, self.pred_load, self.rtt) for a in range(3)]
            feats.append([self.pred_load[i], self.pred_load[j]] + [min(l / self.c["deadline_ms"], 5) for l in lat]
                         + [float(np.argmin(lat)) / 2])
        return np.array(feats)

    def _obs(self):
        prev = self.tr[self.t - 1]                   # last measured load
        tod = 2 * np.pi * (self.t % 288) / 288
        obs = []
        for i in range(self.K):
            j = (i + 1) % self.K
            obs.append([prev[i], prev[j], self.Q[i] / 20, self.Q[j] / 20, self.W[i] / 20, self.D[i] / 40,
                        self.rtt / 100, np.sin(tod), np.cos(tod), self.Q.mean() / 20])
        obs = np.array(obs, dtype=np.float32)
        if self.use_twin:
            obs = np.concatenate([obs, self._twin_features().astype(np.float32)], 1)
        return obs

    def action_mask(self):
        """Twin what-if: forbid options predicted to breach the SLA by mask_factor (keep ≥1)."""
        m = np.ones((self.K, N_ACT), bool)
        if not (self.use_twin and self.c["twin_mask"]): return m
        for i in range(self.K):
            for a in range(N_ACT):
                l = self.est_latency(i, a // 3, self.f[a % 3], self.pred_load, self.rtt)
                m[i, a] = l <= self.c["mask_factor"] * self.c["deadline_ms"]
            if not m[i].any(): m[i] = True
        return m

    def step(self, actions):
        load = self.tr[self.t]                         # true (hidden) load during execution
        c = self.c
        lat, en, viol, rew = np.zeros(self.K), np.zeros(self.K), np.zeros(self.K), np.zeros(self.K)
        add = np.zeros(self.K)
        for i in range(self.K):
            tgt, f = actions[i] // 3, self.f[actions[i] % 3]
            lat[i] = self.est_latency(i, tgt, f, load, self.rtt)
            en[i] = self.energy(tgt, f, self.W[i], self.D[i])
            if tgt == 0: add[i] += self.W[i]
            elif tgt == 1: add[(i + 1) % self.K] += self.W[i]
            viol[i] = float(lat[i] > c["deadline_ms"])
            rew[i] = -(c["w_lat"] * min(lat[i] / c["deadline_ms"], 5) + c["w_energy"] * en[i] / 10 + c["w_sla"] * viol[i])
        served = np.array([self.avail(j, load, 1.0) for j in range(self.K)])
        self.Q = np.clip(self.Q + add - served, 0, 200)
        # WAN RTT: base + congestion burst
        self.rtt = c["wan_rtt"] * (1 + 0.5 * self.rng.random()) + (80 if self.rng.random() < 0.05 else 0)
        self.t += 1
        done = self.t - self.t0 >= c["steps_per_ep"]
        self._arrivals()
        info = dict(lat=lat, energy=en, viol=viol, util=load)
        return self._obs(), rew, done, info

## 5. Agents: Double-DQN per edge node + FedAvg

In [ ]:
# ==== 5. Agents ====
class QNet(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, N_ACT))
    def forward(self, x): return self.net(x)

class DQNAgent:
    def __init__(self, d, cfg):
        self.c = cfg
        self.q = QNet(d, cfg["hidden"]).to(DEVICE); self.tq = copy.deepcopy(self.q)
        self.opt = torch.optim.Adam(self.q.parameters(), lr=cfg["lr"])
        n = cfg["buffer"]
        self.S = np.zeros((n, d), np.float32); self.S2 = np.zeros((n, d), np.float32)
        self.A = np.zeros(n, np.int64); self.R = np.zeros(n, np.float32); self.Dn = np.zeros(n, np.float32)
        self.M2 = np.ones((n, N_ACT), bool)
        self.ptr, self.size, self.steps = 0, 0, 0

    def act(self, s, eps, mask):
        if random.random() < eps:
            return int(np.random.choice(np.flatnonzero(mask)))
        with torch.no_grad():
            q = self.q(torch.tensor(s, device=DEVICE).unsqueeze(0)).cpu().numpy()[0]
        q[~mask] = -1e9
        return int(q.argmax())

    def store(self, s, a, r, s2, d, m2):
        i = self.ptr
        self.S[i], self.A[i], self.R[i], self.S2[i], self.Dn[i], self.M2[i] = s, a, r, s2, d, m2
        self.ptr = (i + 1) % len(self.S); self.size = min(self.size + 1, len(self.S))

    def learn(self):
        if self.size < self.c["batch"] * 4: return None
        idx = np.random.randint(0, self.size, self.c["batch"])
        t = lambda x: torch.tensor(x[idx], device=DEVICE)
        s, a, r, s2, d, m2 = t(self.S), t(self.A), t(self.R), t(self.S2), t(self.Dn), t(self.M2)
        with torch.no_grad():
            qn = self.q(s2).masked_fill(~m2, -1e9)
            a2 = qn.argmax(1, keepdim=True)                                   # Double DQN
            y = r + self.c["gamma"] * (1 - d) * self.tq(s2).gather(1, a2).squeeze(1)
        q = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(q, y)
        self.opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(self.q.parameters(), 10); self.opt.step()
        self.steps += 1
        if self.steps % self.c["target_every"] == 0: self.tq.load_state_dict(self.q.state_dict())
        return loss.item()

def federated_average(agents):
    sizes = [ag.size for ag in agents]
    g = fedavg_state([ag.q.state_dict() for ag in agents], sizes)
    for ag in agents:
        ag.q.load_state_dict(g); ag.tq.load_state_dict(g)

## 6. Training loop (MADRL variants) and heuristic baselines

In [ ]:
# ==== 6. Train / evaluate ====
def train_marl(cfg, use_twin, federated, label):
    set_seed(cfg["seed"])
    env = CloudEdgeEnv(TRAIN_TRACE, cfg, TWIN, use_twin, seed=cfg["seed"])
    agents = [DQNAgent(env.obs_dim, cfg) for _ in range(env.K)]
    decay_eps = int(cfg["episodes"] * cfg["eps_decay_frac"])
    curve, t0 = [], time.time()
    for ep in range(cfg["episodes"]):
        eps = max(cfg["eps_end"], cfg["eps_start"] - (cfg["eps_start"] - cfg["eps_end"]) * ep / max(1, decay_eps))
        s = env.reset(); mask = env.action_mask(); done, ep_r, step = False, 0.0, 0
        while not done:
            acts = [agents[i].act(s[i], eps, mask[i]) for i in range(env.K)]
            s2, r, done, info = env.step(acts); mask2 = env.action_mask()
            for i in range(env.K):
                agents[i].store(s[i], acts[i], r[i], s2[i], float(done), mask2[i])
            if step % cfg["learn_every"] == 0:
                for ag in agents: ag.learn()
            s, mask, ep_r, step = s2, mask2, ep_r + r.mean(), step + 1
        if federated and (ep + 1) % cfg["fed_every"] == 0:
            federated_average(agents)
        curve.append(ep_r / cfg["steps_per_ep"])
        if (ep + 1) % 25 == 0:
            print(f"[{label}] ep {ep+1}/{cfg['episodes']}  avg reward {np.mean(curve[-25:]):.3f}  eps {eps:.2f}  ({time.time()-t0:.0f}s)")
    if federated: federated_average(agents)
    return agents, curve

def make_policy(kind, agents=None):
    def pol(env, s, mask):
        K = env.K
        if kind == "random": return list(np.random.randint(0, N_ACT, K))
        if kind == "local": return [2] * K                         # local, max freq
        if kind == "cloud": return [8] * K                         # cloud
        if kind == "greedy":                                       # uses last measured load (no twin)
            prev = env.tr[env.t - 1]
            return [int(np.argmin([env.est_latency(i, a, 1.0, prev, env.rtt) for a in range(3)])) * 3 + 2 for i in range(K)]
        return [agents[i].act(s[i], 0.0, mask[i]) for i in range(K)]
    return pol

def evaluate(policy, cfg, use_twin, n_eps):
    env = CloudEdgeEnv(TEST_TRACE, cfg, TWIN, use_twin, seed=cfg["seed"] + 999)
    starts = np.linspace(cfg["hist_len"] + 1, len(TEST_TRACE) - cfg["steps_per_ep"] - 3, n_eps).astype(int)
    L, E, V, R = [], [], [], []
    for st in starts:
        s = env.reset(start=st); mask = env.action_mask(); done = False
        while not done:
            a = policy(env, s, mask)
            s, r, done, info = env.step(a); mask = env.action_mask()
            L += list(info["lat"]); E += list(info["energy"]); V += list(info["viol"]); R += list(r)
    L = np.array(L)
    return dict(latency_ms=L.mean(), p95_latency_ms=np.percentile(L, 95), energy=np.mean(E),
                sla_violation_pct=100 * np.mean(V), reward=np.mean(R)), L

In [ ]:
RESULTS, CURVES, LAT_DIST, AGENTS = {}, {}, {}, {}
for name, kind in [("Random", "random"), ("Local-only", "local"), ("Cloud-only", "cloud"), ("Greedy", "greedy")]:
    RESULTS[name], LAT_DIST[name] = evaluate(make_policy(kind), CFG, False, CFG["eval_episodes"])
    print(name, {k: round(v, 3) for k, v in RESULTS[name].items()})

VARIANTS = [("MADRL (indep.)", False, False), ("Fed-MADRL", False, True),
            ("DT-MADRL", True, False), ("CDT-FedMARL (proposed)", True, True)]
for name, tw, fed in VARIANTS:
    agents, curve = train_marl(CFG, tw, fed, name)
    AGENTS[name], CURVES[name] = agents, curve
    RESULTS[name], LAT_DIST[name] = evaluate(make_policy("rl", agents), CFG, tw, CFG["eval_episodes"])
    print(name, {k: round(v, 3) for k, v in RESULTS[name].items()})

## 7. Results: tables and figures

In [ ]:
res_df = pd.DataFrame(RESULTS).T.round(3)
res_df.to_csv("results/table_main_results.csv")
print(res_df)

def smooth(x, w=10): return np.convolve(x, np.ones(w) / w, mode="valid") if len(x) >= w else np.array(x)

plt.figure(figsize=(7, 4))
for n, c in CURVES.items(): plt.plot(smooth(c), label=n)
plt.xlabel("episode"); plt.ylabel("avg reward per slot"); plt.title("Training convergence")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("results/fig_convergence.png", dpi=300); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, col, yl in zip(ax, ["latency_ms", "energy", "sla_violation_pct"], ["Avg latency (ms)", "Energy (J/slot)", "SLA violation (%)"]):
    colors = ["#bbbbbb"] * 4 + ["#7fa7d9", "#4c78a8", "#f2a65a", "#d62728"]
    a.bar(range(len(res_df)), res_df[col], color=colors[:len(res_df)])
    a.set_xticks(range(len(res_df))); a.set_xticklabels(res_df.index, rotation=45, ha="right", fontsize=8)
    a.set_ylabel(yl); a.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("results/fig_comparison.png", dpi=300); plt.show()

plt.figure(figsize=(7, 4))
for n, L in LAT_DIST.items():
    xs = np.sort(L); plt.plot(xs, np.arange(1, len(xs) + 1) / len(xs), label=n, lw=1.2)
plt.axvline(CFG["deadline_ms"], c="k", ls="--", lw=.8)
plt.xscale("log"); plt.xlabel("latency (ms, log)"); plt.ylabel("CDF"); plt.title("Latency CDF (test trace)")
plt.legend(fontsize=7); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("results/fig_latency_cdf.png", dpi=300); plt.show()

# Action distribution of the proposed policy
env = CloudEdgeEnv(TEST_TRACE, CFG, TWIN, True, seed=1); s = env.reset(); m = env.action_mask(); cnt = np.zeros(N_ACT)
pol = make_policy("rl", AGENTS["CDT-FedMARL (proposed)"]); done = False
while not done:
    a = pol(env, s, m); cnt += np.bincount(a, minlength=N_ACT); s, _, done, _ = env.step(a); m = env.action_mask()
lab = [f"{t}\nf={f}" for t in ["Local", "Nbr", "Cloud"] for f in CFG["f_levels"]]
plt.figure(figsize=(8, 3)); plt.bar(lab, cnt / cnt.sum()); plt.ylabel("share"); plt.title("CDT-FedMARL action distribution")
plt.tight_layout(); plt.savefig("results/fig_action_dist.png", dpi=300); plt.show()

## 8. (Optional) Scalability study — vary number of edge agents
Set `RUN_SCALABILITY = True`. It retrains the proposed model for each K, so it takes longer.

In [ ]:
RUN_SCALABILITY = False
if RUN_SCALABILITY:
    rows = []
    for K in [4, 8, 12]:
        cfg = dict(CFG, n_edge=K, episodes=max(60, CFG["episodes"] // 2))
        full = load_trace(dict(cfg)); sp = int(0.7 * len(full))
        TRAIN_TRACE, TEST_TRACE = full[:sp], full[sp:]
        for name, tw, fed in [("Fed-MADRL", False, True), ("CDT-FedMARL (proposed)", True, True)]:
            ag, _ = train_marl(cfg, tw, fed, f"{name} K={K}")
            r, _ = evaluate(make_policy("rl", ag), cfg, tw, cfg["eval_episodes"])
            rows.append(dict(K=K, method=name, **r))
    sc = pd.DataFrame(rows); sc.to_csv("results/table_scalability.csv", index=False); print(sc.round(3))
    TRAIN_TRACE, TEST_TRACE = TRACE[:split], TRACE[split:]   # restore

## 9. Save everything

In [ ]:
torch.save({n: [a.q.state_dict() for a in ags] for n, ags in AGENTS.items()}, "results/agents.pt")
torch.save(TWIN.state_dict(), "results/twin_gru.pt")
json.dump({k: v for k, v in CFG.items()}, open("results/config.json", "w"), indent=2)
subprocess.run("zip -qr results.zip results", shell=True)
try:
    from google.colab import files; files.download("results.zip")
except Exception:
    print("results.zip written to working directory")